# Delta Table Basics Examples

This notebook introduces the basic lifecycle of a Delta table in Databricks.

The flow covers:

- creating a Delta table
- reading table data
- inspecting table history
- checking time travel

In [ ]:
from pyspark.sql import functions as F

In [ ]:
target_table = "main.demo.delta_table_basics"

base_data = [
    (1, "Asha", 1200.0),
    (2, "Miguel", 950.0),
    (3, "Lina", 780.0)
]

base_df = spark.createDataFrame(base_data, ["customer_id", "customer_name", "spend"])

## Create a Delta table

Writing with format `delta` creates a Delta table with data files and a transaction log.

In [ ]:
base_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
print(f"Created or replaced {target_table}")

In [ ]:
display(spark.table(target_table).orderBy("customer_id"))

## Create a second version

Every write creates a new Delta version that can later be inspected.

In [ ]:
incremental_df = spark.createDataFrame([(4, "Noah", 640.0)], ["customer_id", "customer_name", "spend"])
incremental_df.write.format("delta").mode("append").saveAsTable(target_table)

In [ ]:
display(spark.table(target_table).orderBy("customer_id"))

## Table history

The transaction log records each committed version of the table.

In [ ]:
history_df = spark.sql(f"DESCRIBE HISTORY {target_table}")
display(history_df)

## Time travel

Because Delta keeps table versions, you can query an older version directly.

In [ ]:
version_zero_df = spark.sql(f"SELECT * FROM {target_table} VERSION AS OF 0")
display(version_zero_df.orderBy("customer_id"))

## Practical notes

- A Delta table combines data files with a `_delta_log` transaction log.
- Every write creates a new version.
- History and time travel make debugging and audit easier.
- Delta tables are the foundation for merge, CDC, and medallion-style patterns.